In [1]:
from lightgbm import LGBMClassifier
import sys
import os
import time
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))
from cargar_datos import cargar_dataset_recontacto

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve, accuracy_score,
    precision_score, recall_score, f1_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

sns.set_theme(style="whitegrid")

# KPI -> filtro (None = todos los casos)
KPIS = {
    'KPI01 Recontacto general':            None,
    'KPI02 Motivos criticos':              'motivos_criticos',
    'KPI07 Historial de casos':            'historial',
    'KPI08 Canales baja resolucion':       'canales_baja_resolucion',
    'KPI10 Nocturno y fin de semana':      'nocturno_finde',
}

COLS_FUERA = ['id_interaccion', 'fecha_contacto', 'es_recontacto']
ESCALADOS = ['1. Regresión Logística', '6. KNN (k=10)']

os.makedirs('../informe_figuras', exist_ok=True)
os.makedirs('../informe_tablas', exist_ok=True)
print("✅ Librerías cargadas y KPIs definidos")

✅ Librerías cargadas y KPIs definidos


In [2]:
def filtrar(df, filtro):
    if filtro == 'motivos_criticos':
        d = df[(df.motivo_falla == 1) | (df.motivo_queja == 1)]
    elif filtro == 'historial':
        d = df[df.tiene_historial == 1]
    elif filtro == 'canales_baja_resolucion':
        d = df[(df.canal_red_social == 1) | (df.canal_correo == 1)]
    elif filtro == 'nocturno_finde':
        d = df[(df.turno_nocturno == 1) | (df.turno_fin_semana == 1)]
    else:
        d = df
    return d.reset_index(drop=True)


def crear_modelos(ratio):
    return {
        '1. Regresión Logística': LogisticRegression(
            max_iter=1000, class_weight='balanced', random_state=42),
        '2. Árbol de Decisión': DecisionTreeClassifier(
            max_depth=8, min_samples_split=20, min_samples_leaf=10,
            class_weight='balanced', random_state=42),
        '3. Random Forest': RandomForestClassifier(
            n_estimators=300, max_depth=10, min_samples_split=10,
            class_weight='balanced', random_state=42, n_jobs=-1),
        '4. XGBoost': XGBClassifier(
            n_estimators=600, max_depth=5, learning_rate=0.05,
            scale_pos_weight=ratio, random_state=42,
            eval_metric='auc', verbosity=0),
        '5. LightGBM': LGBMClassifier(
            n_estimators=600, max_depth=5, learning_rate=0.05,
            scale_pos_weight=ratio, random_state=42, verbose=-1),
        '6. KNN (k=10)': KNeighborsClassifier(
            n_neighbors=10, weights='distance', n_jobs=-1),
        '7. Naive Bayes': GaussianNB(),
    }

print("✅ Funciones listas")

✅ Funciones listas


In [3]:
df_base = cargar_dataset_recontacto()

res_comp, res_final, res_coef = [], [], []

for kpi, filtro in KPIS.items():
    print("\n" + "=" * 80)
    print(f"▶ {kpi}")
    print("=" * 80)

    d = filtrar(df_base, filtro)
    X = d.drop(columns=COLS_FUERA, errors='ignore')
    y = d['es_recontacto']
    X = X.astype({c: 'int8' for c in X.select_dtypes(include=['bool']).columns}).fillna(0)

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, random_state=42, stratify=y)

    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)

    ratio = (y_train == 0).sum() / (y_train == 1).sum()

    # ---- Comparación de los 7 modelos (evidencia) ----
    lr_guardado = None
    for nombre, modelo in crear_modelos(ratio).items():
        a, b = (X_train_s, X_test_s) if nombre in ESCALADOS else (X_train, X_test)
        modelo.fit(a, y_train)
        pred = modelo.predict(b)
        proba = modelo.predict_proba(b)[:, 1]
        res_comp.append({
            'KPI': kpi, 'Modelo': nombre,
            'Accuracy': accuracy_score(y_test, pred),
            'Precision': precision_score(y_test, pred),
            'Recall': recall_score(y_test, pred),
            'F1': f1_score(y_test, pred),
            'ROC-AUC': roc_auc_score(y_test, proba),
        })
        if nombre == '1. Regresión Logística':
            lr_guardado, lr_pred, lr_proba = modelo, pred, proba

    # ---- Modelo final: Regresión Logística ----
    cm = confusion_matrix(y_test, lr_pred)
    tn, fp, fn, tp = cm.ravel()
    acc = accuracy_score(y_test, lr_pred)
    auc = roc_auc_score(y_test, lr_proba)

    res_final.append({
        'KPI': kpi,
        'Registros': len(d),
        'Tasa recontacto %': round(y.mean() * 100, 2),
        'Train': len(X_train), 'Test': len(X_test),
        'Variables': X.shape[1],
        'Accuracy': round(acc, 4),
        'Precision': round(precision_score(y_test, lr_pred), 4),
        'Recall': round(recall_score(y_test, lr_pred), 4),
        'F1': round(f1_score(y_test, lr_pred), 4),
        'ROC-AUC': round(auc, 4),
        'VN': tn, 'FP': fp, 'FN': fn, 'VP': tp,
    })

    coef = pd.DataFrame({
        'KPI': kpi,
        'Variable': X.columns,
        'Coeficiente': lr_guardado.coef_[0],
    })
    coef['Odds ratio'] = np.exp(coef['Coeficiente'])
    coef['Abs'] = coef['Coeficiente'].abs()
    coef = coef.sort_values('Abs', ascending=False)
    res_coef.append(coef.drop(columns='Abs'))
    top = coef.head(10)

    # ---- Gráficos ----
    fig, axes = plt.subplots(1, 3, figsize=(20, 5.5))
    ConfusionMatrixDisplay(cm, display_labels=['No recontacta', 'Recontacta']).plot(
        ax=axes[0], cmap='Blues', colorbar=False, values_format='d')
    axes[0].set_title(f'Matriz de confusión\n{kpi}')

    fpr, tpr, _ = roc_curve(y_test, lr_proba)
    axes[1].plot(fpr, tpr, lw=2, label=f'Regresión Logística (AUC = {auc:.3f})')
    axes[1].plot([0, 1], [0, 1], 'k--', lw=1, label='Azar')
    axes[1].set_xlabel('Tasa de falsos positivos')
    axes[1].set_ylabel('Tasa de verdaderos positivos')
    axes[1].set_title('Curva ROC')
    axes[1].legend(loc='lower right')

    t = top.sort_values('Coeficiente')
    colores = ['#d62728' if c > 0 else '#2ca02c' for c in t['Coeficiente']]
    axes[2].barh(t['Variable'], t['Coeficiente'], color=colores)
    axes[2].axvline(0, color='black', lw=0.8)
    axes[2].set_title('Top 10 coeficientes\n(rojo: sube la prob. de recontacto, verde: la baja)')
    axes[2].set_xlabel('Coeficiente (variables estandarizadas)')

    plt.tight_layout()
    nombre_png = kpi.split(' ')[0] + '_modelo_final.png'
    plt.savefig(f'../informe_figuras/{nombre_png}', dpi=150, bbox_inches='tight')
    plt.show()

    print(f"Accuracy {acc:.4f} | AUC {auc:.4f} | VN={tn} FP={fp} FN={fn} VP={tp}")
    print(top[['Variable', 'Coeficiente', 'Odds ratio']].round(4).to_string(index=False))

print("\n✅ Los 5 KPIs fueron procesados")

📊 CARGANDO DATASET DE RECONTACTO (KPI 1)
✅ Conexión exitosa a SQL Server
✅ Consulta ejecutada: 11,436 registros, 34 columnas

📏 Dimensiones: 11,436 filas × 34 columnas

🎯 Distribución del target:
es_recontacto
0    6687
1    4749
Name: count, dtype: int64

📈 Tasa de recontacto: 41.53%

▶ KPI01 Recontacto general


<Figure size 2000x550 with 3 Axes>

Accuracy 0.8042 | AUC 0.8803 | VN=1069 FP=269 FN=179 VP=771
            Variable  Coeficiente  Odds ratio
      transferencias       1.5146      4.5476
   casos_previos_30d       0.9228      2.5162
          espera_seg       0.9167      2.5011
       cola_reclamos      -0.4004      0.6700
   es_duracion_corta       0.3706      1.4486
        duracion_seg       0.3558      1.4273
usuario_adulto_mayor       0.3336      1.3960
     usuario_empresa      -0.2846      0.7523
        cola_soporte      -0.2392      0.7873
    cola_informacion       0.2308      1.2596

▶ KPI02 Motivos criticos


<Figure size 2000x550 with 3 Axes>

Accuracy 0.8130 | AUC 0.8926 | VN=449 FP=99 FN=91 VP=377
            Variable  Coeficiente  Odds ratio
      transferencias       1.5790      4.8499
          espera_seg       0.9757      2.6531
   casos_previos_30d       0.9349      2.5469
       cola_reclamos      -0.3944      0.6741
     usuario_empresa      -0.3448      0.7084
usuario_adulto_mayor       0.3415      1.4071
        duracion_seg       0.3166      1.3725
   es_duracion_corta       0.2828      1.3268
        cola_soporte      -0.2759      0.7589
    cola_informacion       0.2619      1.2994

▶ KPI07 Historial de casos


<Figure size 2000x550 with 3 Axes>

Accuracy 0.8063 | AUC 0.8744 | VN=523 FP=120 FN=131 VP=522
            Variable  Coeficiente  Odds ratio
      transferencias       1.4965      4.4660
          espera_seg       0.9149      2.4966
   casos_previos_30d       0.8173      2.2644
       cola_reclamos      -0.3612      0.6968
   es_duracion_corta       0.3490      1.4177
        duracion_seg       0.3402      1.4053
usuario_adulto_mayor       0.2812      1.3248
     usuario_empresa      -0.2586      0.7721
    cola_informacion       0.2540      1.2892
      turno_nocturno       0.2355      1.2655

▶ KPI08 Canales baja resolucion


<Figure size 2000x550 with 3 Axes>

Accuracy 0.7975 | AUC 0.8837 | VN=474 FP=110 FN=102 VP=361
            Variable  Coeficiente  Odds ratio
      transferencias       1.6419      5.1649
          espera_seg       0.8855      2.4242
   casos_previos_30d       0.8749      2.3986
       cola_reclamos      -0.4158      0.6598
   es_duracion_corta       0.4013      1.4938
usuario_adulto_mayor       0.3623      1.4367
        duracion_seg       0.3457      1.4130
     usuario_empresa      -0.3064      0.7361
    cola_informacion       0.2635      1.3015
        cola_soporte      -0.2315      0.7933

▶ KPI10 Nocturno y fin de semana


<Figure size 2000x550 with 3 Axes>

Accuracy 0.8094 | AUC 0.8951 | VN=412 FP=102 FN=73 VP=331
            Variable  Coeficiente  Odds ratio
      transferencias       1.3619      3.9035
          espera_seg       1.0003      2.7192
   casos_previos_30d       0.9477      2.5798
   es_duracion_corta       0.4072      1.5025
        duracion_seg       0.3901      1.4771
usuario_adulto_mayor       0.3386      1.4030
       cola_reclamos      -0.3343      0.7158
     usuario_empresa      -0.2552      0.7747
    canal_red_social       0.2342      1.2640
    cola_informacion       0.2281      1.2562

✅ Los 5 KPIs fueron procesados


In [4]:
df_comp = pd.DataFrame(res_comp)
df_final = pd.DataFrame(res_final)
df_coef = pd.concat(res_coef, ignore_index=True)

# Ganador por KPI (mayor ROC-AUC)
ganadores = df_comp.loc[df_comp.groupby('KPI')['ROC-AUC'].idxmax(), ['KPI', 'Modelo', 'Accuracy', 'ROC-AUC']]
ganadores = ganadores.reset_index(drop=True)

print("=" * 90)
print("🏆 MODELO GANADOR POR KPI (criterio: mayor ROC-AUC)")
print("=" * 90)
print(ganadores.round(4).to_string(index=False))

print("\n" + "=" * 90)
print("📊 ACCURACY DE LOS 7 MODELOS POR KPI")
print("=" * 90)
print(df_comp.pivot(index='Modelo', columns='KPI', values='Accuracy').round(4).to_string())

print("\n" + "=" * 90)
print("📊 ROC-AUC DE LOS 7 MODELOS POR KPI")
print("=" * 90)
print(df_comp.pivot(index='Modelo', columns='KPI', values='ROC-AUC').round(4).to_string())

print("\n" + "=" * 90)
print("📋 RESUMEN DEL MODELO FINAL (REGRESIÓN LOGÍSTICA)")
print("=" * 90)
print(df_final.to_string(index=False))

# Guardar evidencia en archivos
df_comp.round(4).to_csv('../informe_tablas/comparacion_modelos_por_kpi.csv', index=False)
df_final.to_csv('../informe_tablas/resumen_modelo_final.csv', index=False)
df_coef.round(4).to_csv('../informe_tablas/coeficientes_por_kpi.csv', index=False)
print("\n✅ Tablas guardadas en ../informe_tablas y figuras en ../informe_figuras")

🏆 MODELO GANADOR POR KPI (criterio: mayor ROC-AUC)
                           KPI                 Modelo  Accuracy  ROC-AUC
      KPI01 Recontacto general 1. Regresión Logística    0.8042   0.8803
        KPI02 Motivos criticos 1. Regresión Logística    0.8130   0.8926
      KPI07 Historial de casos 1. Regresión Logística    0.8063   0.8744
 KPI08 Canales baja resolucion 1. Regresión Logística    0.7975   0.8837
KPI10 Nocturno y fin de semana 1. Regresión Logística    0.8094   0.8951

📊 ACCURACY DE LOS 7 MODELOS POR KPI
KPI                     KPI01 Recontacto general  KPI02 Motivos criticos  KPI07 Historial de casos  KPI08 Canales baja resolucion  KPI10 Nocturno y fin de semana
Modelo                                                                                                                                                           
1. Regresión Logística                    0.8042                  0.8130                    0.8063                         0.7975                     